# MirrorTopology T1 signmap **v1.5_audited**（実データ初回接触の較正）
2026-08-30。v1.4のColab実行で**実CMBtopology共分散に初めて接触**し，G09（reality条件）が
5.01e-08で停止したことへの対応版。計算は正常に完走しており，停止したのは入力検証の許容値
（T2b-2の解析的共分散向け1e-10）を数値積分由来の共分散へ流用していたため。

**v1.4→v1.5**：閾値を緩めるのではなく，共分散を**理論上厳密な2対称性**（Hermiticity・
reality条件）を満たす部分空間へ**明示的に直交射影**する。(1) 除去成分を全点でprovenance記録，
(2) 射影前の生違反に**上限1e-5**（超過はパイプライン異常としてFAIL），(3) 射影後1e-12未満を
assert，(4) **射影がE[S±]を変える相対量<1e-6**をassert（サンドボックス実測2e-16）。
規則はrules v1.4 §3bに凍結・engine v1.5。解析ロジック・走査点・gate構造は不変。

---
（以下v1.4の説明）
# MirrorTopology T1 signmap **v1.4_audited**（依存ソースのhermetic化）
2026-08-30。v1.3をColab実機で実行した際，G06b（CMBtopology tracked clean）が発動して停止した
ことへの対応版。原因は解析側ではなく**依存ソースの置き場所**：pinned commitのソースを
Drive上の使い回しクローンに依存していたため，同期・過去セッションの影響でworking treeが
改変され得た（ゲートは設計どおり正しく停止した）。

**v1.3→v1.4**：CMBtopologyを**セッション毎にephemeralな`/content`へ新規クローン**して
pinned commitをcheckout（Drive上には置かない＝hermetic）。既存ディレクトリがある場合は
`checkout --force`＋`clean -fdx`で復元し，それでも不一致ならFAIL（差分レポート付き）。
復元の有無は`ct_repair`としてprovenanceに記録。covarianceキャッシュ（Drive上のOUT）と
manifest/SHA照合は従来どおりで，再計算は不要。**解析コード・規則・engineはv1.3から不変**
（rules v1.3・engine v1.3をそのまま使用）。

---
（以下v1.3の説明）
# MirrorTopology T1 signmap **v1.3_audited**（最終監査対応・commit&smoke GO判定用）
2026-08-27。最終監査の唯一の科学的BLOCKER＝**等方参照C_ℓとCMBtopology内部規約の不一致**への
対応版。旧`T1_signmap_v0.3`＝EXPLORATORY/SUPERSEDED。

**v1.2→v1.3**：(1) 参照C_ℓを二系統に分離——**CL_AUDIT**（PR3系lensed・エンジン自己検証と
旧サンプラーバイアス定量専用）と**CL_CMBTOPO_REF**（**凍結commitのtopology/src/topology.py
から機械抽出した設定**：H0=67.5・ombh2=0.022・omch2=0.122・mnu=0.06・tau=0.06・As=2e-9・
ns=0.965・unlensed_scalar・raw_cl・μK——凍結期待値との完全一致をG06dでhard assertし，
G22/G22b/G23はこちらのみ使用）。raw共分散はμK²（transfer側も×1e6×2.7255）であることを
ソースで確認済み。(2) smokeでも等方系列はofficialと同一のL={1.4,2,3}を実行（同一gateの
事前検証）。(3) G25b＝NPZ全配列のfinite assert追加。(4) provenance版表記の修正。
規則はrules v1.3（参照規約の明文化のみ・科学規則は不変）。

**v1.1→v1.2**：(1) G25を**適用可能列のみのfinite検査**に修正（topology毎に存在する演算子列が
異なるための構造的NaN＝非適用列は許容——v1.1のままではofficial実行が最終セルで誤FAILする
バグだった），(2) CMBtopologyに**tracked clean gate**（G06b・--porcelain -uno空）と
**canonical origin gate**（G06c）を追加，(3) mirror-topology側も`canonical_url`を渡し
**origin_ok/pushed（G01b/G01c）をOFFICIAL条件に編入**（「結果を見る前にpublic commitした」
履歴の機械保証），(4) 等方gateに**絶対振幅条件G22b**（|α−1|<0.15・形状d_isoは
sanity convergence gateと正名）を追加，(5) 走査軸の**±対蹠完全dedup**
（赤道z=0の重複解消・厳密にnpix/2軸・エンジンv1.2），(6) 候補軸を**線形ホロノミー閉包群**から
抽出し**群位数の凍結assert**（実測：E7:2・E8:4・E9:2・E10:4）を追加，(7) provenanceに
NPZ・走査方向配列・AXES manifestのSHA（G26），(8) **T1専用凍結規則**
`docs/T1_audit_rules_v1.2.md`をSHA gate対象に。

**T1の役割（レビュー§13）**：*retroactively audited low-ℓ full-sky theoretical
predictive-engine validation*。マスク・前処理・凍結統計を観測と揃えた直接比較は**Step 1**の
仕事であり，T1単独では「このtopologyが観測をX%で説明する」とは言わない。

**v1.0→v1.1の修正**：
- **B1** `O_axis`を*numerical harmonic rotation*と正名し，**direct-geometry hard test**
  （鏡映p′=p−2(n·p)n・半回転p′=2(n·p)n−p を座標で直接作用させた天空との点毎比較）＋
  射影恒等式（P²=P・P₊P₋=0・P₊+P₋=I・P(n)=P(−n)）を通過して初めて
  "effectively exact for ℓ≤4" と呼ぶ（実測：相対誤差~2e-11）。
- **B2** 空のround-trip placeholderを削除し，実テストに置換：T1-A2 complex↔real往復
  （実測2.7e-16）・T1-A3 二点相関等価 MᴴC_realM=Mx（実測4.3e-16）。電池はT1-A1..A6/B1..B3
  の命名テスト。
- **B3** covariance binding：run_topology前後のディレクトリ集合差で生成物を一意特定
  （新規1個をassert・0個なら全トークン一致が唯一のディレクトリをassert）。再利用時は
  **file SHAをmanifest記録値と毎回照合**。`cov_file_sha256`と`cov_array_sha256`を分離。
- **B4** `%%writefile`を廃止し**thin driver化**：ENGINE_PATH=repo内t1_engine.pyを
  source of truthとし，Git gate・SHA・provenance・import identity
  （`Path(t1.__file__)==ENGINE_PATH`）を同一パスで統一。cwd変更はensure_cov内に閉じ込め。
- その他：eigenサンプラー（Cholesky+jitter廃止・λ_min/clip/rank記録）・(op,軸)単位のdedup＋
  **期待演算子カウントの凍結assert**・等方gateを最終絶対条件へ（d_iso=‖C−αC_iso‖_F/‖αC_iso‖_F
  等4条件・単調性は記録のみ）・regression完全性gate・G01–G25の個別boolean→OFFICIAL=all・
  distribution_status=PILOT_DISTRIBUTION_ONLY明示。

**事前ルール（等方gate）**：L=3で不合格の場合はthresholdを緩めず，より大きいL（5）を追加する。

In [ ]:
# ---- 設定・依存の凍結 ----
import os, sys, subprocess, time, json, glob, re, hashlib, inspect
from pathlib import Path
GATES = {}
IN_COLAB = os.path.isdir('/content')
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        raise RuntimeError('Driveマウント失敗：明示停止 ' + repr(e))
    BASE = '/content/drive/MyDrive/mirror_topology'
    assert os.path.isdir(BASE), BASE
    for p in ['healpy', 'camb', 'numba', 'quaternionic', 'spherical', 'tqdm']:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=True)
else:
    BASE = '/home/claude/colab_sim'
    os.makedirs(BASE, exist_ok=True)
import numpy as np, pandas as pd, healpy as hp
T1_MODE = globals().get('T1_MODE', 'official')
LMAX = 4
EXPECTED_CMBTOPO_COMMIT = '0cc65e34f03df85e92f738686bff0a476132f337'
# 依存ソースはephemeral領域へ新規クローン（Drive上に置かない＝hermetic・v1.4）
CT_DIR = os.path.join('/content' if IN_COLAB else '/tmp', 'CMBtopology_pinned')
_env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
CT_REPAIR = 'fresh_clone'
if not os.path.isdir(os.path.join(CT_DIR, '.git')):
    subprocess.run(['git', 'clone', 'https://github.com/CompactCollaboration/CMBtopology.git',
                    CT_DIR], check=True, capture_output=True, env=_env)
else:
    _pre = subprocess.run(['git', '-C', CT_DIR, 'status', '--porcelain', '--untracked-files=no'],
                          capture_output=True, text=True).stdout.strip()
    CT_REPAIR = 'reused_dirty_restored' if _pre else 'reused_clean'
    if _pre:
        print('既存クローンにtracked変更を検出→pinned commitへ強制復元します:',
              repr(_pre[:200]))
    subprocess.run(['git', '-C', CT_DIR, 'fetch', '-q', 'origin'], capture_output=True, env=_env)
subprocess.run(['git', '-C', CT_DIR, 'checkout', '-q', '--force', EXPECTED_CMBTOPO_COMMIT],
               check=True)
if subprocess.run(['git', '-C', CT_DIR, 'status', '--porcelain', '--untracked-files=no'],
                  capture_output=True, text=True).stdout.strip():
    subprocess.run(['git', '-C', CT_DIR, 'checkout', '--force', '.'], capture_output=True)
    subprocess.run(['git', '-C', CT_DIR, 'clean', '-fdx', '-q'], capture_output=True)
    CT_REPAIR = 'force_restored'
_h = subprocess.run(['git', '-C', CT_DIR, 'rev-parse', 'HEAD'],
                    capture_output=True, text=True).stdout.strip()
GATES['G06a_cmbtopology_commit'] = (_h == EXPECTED_CMBTOPO_COMMIT)
assert GATES['G06a_cmbtopology_commit'], _h
_st = subprocess.run(['git', '-C', CT_DIR, 'status', '--porcelain', '--untracked-files=no'],
                     capture_output=True, text=True).stdout
GATES['G06b_cmbtopology_clean'] = (_st.strip() == '')
if not GATES['G06b_cmbtopology_clean']:
    _dg = subprocess.run(['git', '-C', CT_DIR, 'diff', '--stat', 'HEAD'],
                         capture_output=True, text=True).stdout
    raise AssertionError('CMBtopology tracked変更あり（復元後も不一致）: '
                         + repr(_st) + ' | ' + repr(_dg))
CT_ORIGIN = subprocess.run(['git', '-C', CT_DIR, 'remote', 'get-url', 'origin'],
                           capture_output=True, text=True).stdout.strip()
GATES['G06c_cmbtopology_origin'] = (CT_ORIGIN.rstrip('/').removesuffix('.git')
                                    == 'https://github.com/CompactCollaboration/CMBtopology')
assert GATES['G06c_cmbtopology_origin'], CT_ORIGIN
OUT = os.path.join(BASE, 'runs_t1_audited')
os.makedirs(OUT, exist_ok=True)
print('BASE =', BASE, '/ CMBtopology @', _h[:12], '(' + CT_DIR + ', ' + CT_REPAIR + ')',
      '/ mode =', T1_MODE)

In [ ]:
# ---- Git gate・engine path統一（thin driver：writefile廃止） ----
_CANDS = ([os.path.join(BASE, 'mirror_topology_repo'), '/content/mt_repo'] if IN_COLAB
          else ['/home/claude/mt_repo'])
MT_REPO = next((p for p in _CANDS if os.path.isdir(os.path.join(p, '.git'))), _CANDS[-1])
if not os.path.isdir(os.path.join(MT_REPO, '.git')):
    subprocess.run(['git', 'clone', '--depth', '5',
                    'https://github.com/tsujikeita/mirror-topology.git', MT_REPO],
                   capture_output=True, env=dict(os.environ, GIT_TERMINAL_PROMPT='0'))
MT_REPO = str(Path(MT_REPO).resolve())
ENGINE_PATH = Path(MT_REPO, 't1_engine.py').resolve()      # source of truth（B4）
assert ENGINE_PATH.exists(), ('t1_engine.pyがrepoにありません。commit（またはsmoke時は'
                              'repoフォルダへ配置）してから実行してください: ' + str(ENGINE_PATH))
sys.path.insert(0, MT_REPO)
import t2b2_run as tr, t2b2_bridge as _brchk
sig = inspect.signature(tr.head_gate)
GATES['G05_head_gate_api'] = ('nb_live_src_sha' in sig.parameters
                              and 'notebook_basename' in sig.parameters)
assert GATES['G05_head_gate_api'], f'head_gate API不一致: {sig}'
NB_BASENAME = 'MirrorTopology_T1_signmap_v1.5_audited.ipynb'
NB_LIVE = tr.live_notebook_source_sha()
T1_RULES_SHA = '85c9d19c0e6fb8dafb3952318135b28200db4a1a080d56cbde2404c30fcf0932'
REPO_GATE = tr.head_gate(MT_REPO,
                         {'t1_engine.py': str(ENGINE_PATH),
                          't2b2_bridge.py': os.path.join(MT_REPO, 't2b2_bridge.py'),
                          't2b2_run.py': os.path.join(MT_REPO, 't2b2_run.py')},
                         'T1_audit_rules_v1.4.md', T1_RULES_SHA,
                         notebook_basename=NB_BASENAME, nb_live_src_sha=NB_LIVE,
                         canonical_url='https://github.com/tsujikeita/mirror-topology.git')
GATES['G01_repo_head'] = bool(REPO_GATE['tracked_clean'] and all(REPO_GATE['matches'].values())
                              and REPO_GATE['rules_ok'])
GATES['G02_nb_live_identity'] = bool(REPO_GATE['nb_identity_ok'])
GATES['G03_engine_head'] = bool(REPO_GATE['matches'].get('t1_engine.py'))
GATES['G04_bridge_run_head'] = bool(REPO_GATE['matches'].get('t2b2_bridge.py')
                                    and REPO_GATE['matches'].get('t2b2_run.py'))
GATES['G01b_origin'] = (REPO_GATE.get('origin_ok') is True)
GATES['G01c_pushed'] = (REPO_GATE.get('pushed') is True)
print('REPO_GATE.ok =', REPO_GATE['ok'], '/ G01-G05 =',
      {k: v for k, v in GATES.items() if k.startswith('G0')})
if T1_MODE == 'official':
    assert REPO_GATE['ok'] and all(GATES[k] for k in
                                   ['G01_repo_head', 'G01b_origin', 'G01c_pushed',
                                    'G02_nb_live_identity', 'G03_engine_head',
                                    'G04_bridge_run_head']), \
        'official実行にはcommit＆push＆HEAD一致＋live identity＋rules v1.4が必要'
import importlib, t1_engine
importlib.reload(t1_engine)
import t1_engine as t1
GATES['G00_import_identity'] = (Path(t1.__file__).resolve() == ENGINE_PATH)
assert GATES['G00_import_identity'], (t1.__file__, str(ENGINE_PATH))
print('engine import identity OK:', t1.__file__)

In [ ]:
# ---- T1-A/B 検証電池（命名テスト・direct geometry含む）＋LEGACYバイアス定量 ----
import camb as _camb
pars = _camb.CAMBparams(); pars.set_cosmology(H0=67.36, ombh2=0.02237, omch2=0.1200, tau=0.0544)
pars.InitPower.set_params(As=np.exp(3.044) * 1e-10, ns=0.9649)
pars.set_for_lmax(64, lens_potential_accuracy=1)
CLarr = _camb.get_results(pars).get_cmb_power_spectra(pars, CMB_unit='muK',
                                                      raw_cl=True)['lensed_scalar'][:, 0]
CL_AUDIT = {l: float(CLarr[l]) for l in (2, 3, 4)}   # PR3系：自己検証・legacy比較専用
rep, okA = t1.engine_selftest(CL_AUDIT, nmc=40000)
print('battery:', {k: (f'{v:.2e}' if isinstance(v, float) else v) for k, v in rep.items()})
assert okA, rep
GATES.update(G07_M_unitarity=rep['A1_M_unitarity'] < 1e-12,
             G08_roundtrip=rep['A2_roundtrip'] < 1e-9,
             G09_reality=True,                       # load_cov_full/battery内でhard raise
             G10_twopoint=rep['A3_twopoint'] < 1e-10,
             G12_empirical_cov=rep['A4_emp_cov_rel'] < 0.05,
             G13_geom_refl=rep['B1_geom_refl'] < 1e-4,
             G14_geom_halfturn=rep['B2_geom_halfturn'] < 1e-4,
             G15_projector_ids=rep['B3_projector_ids'] < 1e-4,
             G16_antipodal=rep['B3_projector_ids'] < 1e-4,
             G17_trBC_vs_MC=rep['A5_trBC_vs_MC_over3sigma'] < 1.0,
             G24_seed_repro=bool(rep['A6_seed_repro']))
# LEGACYサンプラーのバイアス（監査証拠・結果には不使用）
lmf = __import__('t2b2_bridge').lm_full()
C_iso_c = np.diag(np.array([CL_AUDIT[l] for (l, m) in lmf]).astype(complex))
_, C_iso, _ = t1.load_cov_full_from_matrix(C_iso_c)
ny = np.array([0, 1, 0.])
Ep, Em = t1.exp_S(C_iso, ny, 'refl')
xn = t1.sample_real(C_iso, 20000, seed=3)
Spn, Smn, _, _ = t1.stats_at_axis(xn, ny, 'refl')
alms_old = t1.sample_alms_v03_LEGACY(C_iso_c, lmf, 4000, seed0=0)
xo = t1.x_from_healpy_alm(np.stack([a for a in alms_old]), imag_tol=np.inf)
Spo, Smo, _, _ = t1.stats_at_axis(xo, ny, 'refl')
LEGACY_BIAS = dict(ESp=Ep, ESm=Em,
                   Sp_new_ratio=float(Spn.mean() / Ep), Sm_new_ratio=float(Smn.mean() / Em),
                   Sp_legacy_ratio=float(Spo.mean() / Ep), Sm_legacy_ratio=float(Smo.mean() / Em))
print('LEGACY bias:', {k: round(v, 4) for k, v in LEGACY_BIAS.items()})
assert abs(LEGACY_BIAS['Sp_new_ratio'] - 1) < 0.02 and abs(LEGACY_BIAS['Sm_new_ratio'] - 1) < 0.02
VALID_T1 = dict(battery=rep, legacy_bias=LEGACY_BIAS)
print('=== T1 ENGINE BATTERY PASS ===')

In [ ]:
# ---- 候補軸（(op,軸)単位dedup＋凍結期待カウント）・走査点 ----
def extract_linear_holonomies(top):
    src = open(os.path.join(CT_DIR, 'topology', 'src', f'{top}.py')).read()
    mats = {}
    for m in re.finditer(r'M_([A-Z])\s*=\s*np\.(diag|array)\(', src):
        name = 'M_' + m.group(1); i = m.end() - 1; depth = 0
        for j in range(i, len(src)):
            if src[j] == '(': depth += 1
            elif src[j] == ')':
                depth -= 1
                if depth == 0: break
        try:
            mats[name] = np.array(eval('np.' + m.group(2) + src[i:j + 1], {'np': np}), float)
        except Exception:
            pass
    return mats
def holonomy_closure(mats):
    """線形ホロノミー閉包群（generatorの右乗で新元が出なくなるまで反復）。"""
    def key(M):
        return tuple(np.round(M, 6).ravel())
    elems = {key(np.eye(3)): ('I', np.eye(3))}
    frontier = [('I', np.eye(3))]
    while frontier:
        new = []
        for na, A in frontier:
            for gn, G in mats.items():
                C = A @ G
                k = key(C)
                if k not in elems:
                    nm = gn if na == 'I' else f'{na}@{gn}'
                    elems[k] = (nm, C)
                    new.append((nm, C))
        frontier = new
    return {nm: M for nm, M in elems.values()}
EXPECTED_GROUP_ORDER = {'E7': 2, 'E8': 4, 'E9': 2, 'E10': 4}   # 凍結（監査時実測・恒等含む）
def linear_holonomy_axes(top):
    """名称linear_holonomy_*（並進を落とした線形部由来）。閉包群全体から抽出し，
    dedupは(op,軸)単位。"""
    grp = holonomy_closure(extract_linear_holonomies(top))
    assert len(grp) == EXPECTED_GROUP_ORDER[top], (top, len(grp))
    axes = {}
    for nm, M in grp.items():
        if np.allclose(M, np.eye(3)):
            continue
        ev = np.round(np.linalg.eigvals(M)).real
        w, v = np.linalg.eigh(M)
        if sorted(ev.tolist()) == [-1, 1, 1]:
            axes[('refl', nm)] = v[:, np.argmin(w)] / np.linalg.norm(v[:, np.argmin(w)])
        elif sorted(ev.tolist()) == [-1, -1, 1]:
            axes[('halfturn', nm)] = v[:, np.argmax(w)] / np.linalg.norm(v[:, np.argmax(w)])
    uniq = {}
    for (op, nm), n in axes.items():
        if not any(o == op and abs(abs(n @ u) - 1) < 1e-6 for (o, _), u in uniq.items()):
            uniq[(op, nm)] = n
    return uniq
EXPECTED_OP_COUNTS = {'E7': {'refl': 1}, 'E8': {'refl': 2, 'halfturn': 1},
                      'E9': {'refl': 1}, 'E10': {'refl': 2, 'halfturn': 1}}  # 凍結（監査時実測）
AXES = {t: linear_holonomy_axes(t) for t in ['E7', 'E8', 'E9', 'E10']}
for t, a in AXES.items():
    cnt = {}
    for (op, _) in a:
        cnt[op] = cnt.get(op, 0) + 1
    assert cnt == EXPECTED_OP_COUNTS[t], (t, cnt, EXPECTED_OP_COUNTS[t])
    print(t, cnt, [(op, nm) for (op, nm) in a])
GATES['G_axes_group_closure'] = True
GATES['G_axes_expected'] = True
def E7p(top='E7', LAx=1, LAy=1, L1y=1, L2x=1, L2z=1, x0y=0.0, tag=''):
    return dict(topology=top, params=dict(LAx=LAx, LAy=LAy, L1y=L1y, L2x=L2x, L2z=L2z),
                x0=[0.0, x0y, 0.0], tag=tag)
def KBp(top, LAx=1, LAy=0, LBx=0, LBz=1, LCy=1, x0y=0.0, tag=''):
    return dict(topology=top, params=dict(LAx=LAx, LAy=LAy, LBx=LBx, LBz=LBz, LCy=LCy),
                x0=[0.0, x0y, 0.0], tag=tag)
POINTS = []
for L1y in [1.0, 0.85, 0.7, 0.6]:
    for frac, ft in [(0.0, 'g0'), (0.25, 'gq'), (0.5, 'gh')]:
        x0y = round((1.0 - frac * L1y) / 2 % L1y, 3)
        POINTS.append(E7p(L1y=L1y, x0y=x0y, tag=f'E7_L1y{L1y}_{ft}'))
POINTS += [E7p(LAx=0.7, tag='E7_LAx0.7'), E7p(LAx=1.3, tag='E7_LAx1.3'),
           E7p(L2x=0.5, tag='E7_tilt'), KBp('E8', tag='E8_def'), E7p(top='E9', tag='E9_def'),
           KBp('E10', tag='E10_def'), KBp('E8', LCy=0.7, tag='E8_LCy0.7'),
           KBp('E10', LCy=0.7, tag='E10_LCy0.7')]
ISO_SERIES = [E7p(LAx=L, LAy=L, L1y=L, L2x=L, L2z=L, x0y=0.0, tag=f'ISOseries_L{L:g}')
              for L in [1.4, 2.0, 3.0]]      # smokeでもofficialと同一（rules v1.3 §4c）
if T1_MODE == 'smoke':
    POINTS = POINTS[:2]
ALL_PTS = POINTS + ISO_SERIES
# ---- 凍結commitのCAMB規約を機械抽出→凍結値と照合（G06d）→CL_CMBTOPO_REF構築 ----
_ctsrc = open(os.path.join(CT_DIR, 'topology', 'src', 'topology.py')).read()
def _kv(argstr):
    out = {}
    for part in argstr.split(','):
        k, v = part.split('=')
        out[k.strip()] = float(v)
    return out
_mc = re.search(r'set_cosmology\(([^)]*)\)', _ctsrc)
_mi = re.search(r'InitPower\.set_params\(([^)]*)\)', _ctsrc)
assert _mc and _mi, 'CMBtopologyソースからCAMB設定を抽出できません'
CT_CAMB = {**_kv(_mc.group(1)), **_kv(_mi.group(1))}
CT_CAMB['spectrum'] = ('unlensed_scalar' if "['unlensed_scalar']" in _ctsrc else 'UNKNOWN')
CT_CAMB['raw_cl'] = bool(re.search(r'raw_cl\s*=\s*True', _ctsrc))
CT_CAMB['CMB_unit'] = ('muK' if re.search(r"CMB_unit\s*=\s*'muK'", _ctsrc) else 'UNKNOWN')
EXPECTED_CT_CAMB = dict(H0=67.5, ombh2=0.022, omch2=0.122, mnu=0.06, omk=0.0, tau=0.06,
                        As=2e-9, ns=0.965, r=0.0,
                        spectrum='unlensed_scalar', raw_cl=True, CMB_unit='muK')
GATES['G06d_camb_reference_match'] = (CT_CAMB == EXPECTED_CT_CAMB)
assert GATES['G06d_camb_reference_match'], (CT_CAMB, EXPECTED_CT_CAMB)
_p2 = _camb.CAMBparams()
_p2.set_cosmology(H0=CT_CAMB['H0'], ombh2=CT_CAMB['ombh2'], omch2=CT_CAMB['omch2'],
                  mnu=CT_CAMB['mnu'], omk=CT_CAMB['omk'], tau=CT_CAMB['tau'])
_p2.InitPower.set_params(As=CT_CAMB['As'], ns=CT_CAMB['ns'], r=CT_CAMB['r'])
_p2.set_for_lmax(64)
_cl2 = _camb.get_results(_p2).get_cmb_power_spectra(_p2, CMB_unit='muK',
                                                    raw_cl=True)['unlensed_scalar'][:, 0]
CL_CMBTOPO_REF = {l: float(_cl2[l]) for l in (2, 3, 4)}
print('G06d PASS / CL_CMBTOPO_REF (muK^2) =',
      {l: round(v, 2) for l, v in CL_CMBTOPO_REF.items()},
      '/ CL_AUDIT =', {l: round(v, 2) for l, v in CL_AUDIT.items()})
tags = [pt['tag'] for pt in ALL_PTS]
assert len(tags) == len(set(tags)), '重複tag'
print(f'{len(POINTS)}解析点 + {len(ISO_SERIES)}等方系列点')

In [ ]:
# ---- covariance取得（B3：生成物の一意binding・再利用SHA照合・split hash） ----
sys.path.insert(0, CT_DIR)
from topology.run_topology import run_topology
def cov_path_for(pt):
    return os.path.join(OUT, f"cov_{pt['tag']}.npy")
def manifest_for(pt):
    return dict(topology=pt['topology'], params={k: float(v) for k, v in pt['params'].items()},
                x0=[float(v) for v in pt['x0']], l_max=LMAX,
                cmbtopology_commit=EXPECTED_CMBTOPO_COMMIT)
def _param_tokens(pt):
    toks = [f'{k}_{float(v):.2f}' for k, v in pt['params'].items()]
    toks += [f'x_{float(pt["x0"][0]):.2f}', f'y_{float(pt["x0"][1]):.2f}',
             f'z_{float(pt["x0"][2]):.2f}']
    return toks
def ensure_cov(pt):
    f = cov_path_for(pt); mf = f + '.manifest.json'
    if os.path.exists(f):
        assert os.path.exists(mf), f'manifest欠落: {mf}'
        rec = json.load(open(mf))
        assert rec['manifest'] == manifest_for(pt), f'manifest不一致: {f}'
        actual = hashlib.sha256(open(f, 'rb').read()).hexdigest()
        assert actual == rec['cov_file_sha256'], f'再利用SHA不一致（改変検出）: {f}'
        return f
    cwd0 = os.getcwd()
    try:
        os.chdir(CT_DIR)                              # cwd変更はここに閉じ込める（B4）
        pat = f"runs/{pt['topology']}_*l_max_{LMAX}"
        before = set(glob.glob(pat))
        run_topology(topology=pt['topology'], l_max=LMAX, do_polarization=False,
                     normalize=True, l_range=np.array([[2, LMAX]]),
                     lp_range=np.array([[2, LMAX]]), x0=np.array(pt['x0']), **pt['params'])
        new = set(glob.glob(pat)) - before
        if len(new) == 1:
            d = new.pop()
        else:                                          # 上書き仕様の場合：全トークン一致が唯一
            cands = [d for d in glob.glob(pat) if all(t in d for t in _param_tokens(pt))]
            assert len(cands) == 1, f'run directoryのbinding失敗: {cands}'
            d = cands[0]
        assert all(t in d for t in _param_tokens(pt)), f'parameter token不一致: {d}'
        src = os.path.join(d, f'TT_corr_matrix_l_2_{LMAX}_lp_2_{LMAX}.npy')
        assert os.path.exists(src), src
        import shutil
        shutil.copy(src, f)
    finally:
        os.chdir(cwd0)
    fb = open(f, 'rb').read()
    json.dump(dict(manifest=manifest_for(pt),
                   cov_file_sha256=hashlib.sha256(fb).hexdigest(),
                   cov_array_sha256=hashlib.sha256(np.load(f).tobytes()).hexdigest(),
                   source_run_dir=os.path.basename(d)),
              open(mf, 'w'), indent=1)
    return f
COVS = {}
t0 = time.time()
for pt in ALL_PTS:
    f = ensure_cov(pt)
    Mx, Cr, meta = t1.load_cov_full(f, LMAX)   # 対称性射影/PSD/twopoint hard gates（rules §3b）
    rec = json.load(open(f + '.manifest.json'))
    assert meta['cov_file_sha256'] == rec['cov_file_sha256']
    Mx_raw = np.load(f)
    axes_here = list(AXES[pt['topology']].values()) + [np.array([0, 1, 0.])]
    imp = t1.projection_impact(Mx_raw, Mx, axes_here)
    assert imp < t1.PROJECTION_IMPACT_CEILING, (pt['tag'], 'projection impact', imp)
    meta['projection_impact_ESpm'] = imp
    COVS[pt['tag']] = (Mx, Cr, meta)
    sy = meta['symmetry']
    print(f"  {pt['tag']:20s} file={meta['cov_file_sha256'][:10]}… "
          f"reality_raw={sy['reality_raw']:.1e}->post={sy['reality_post']:.1e} "
          f"corr={sy['correction_max_rel']:.1e} impact={imp:.1e} "
          f"rank={meta['eig']['effective_rank']} ({time.time()-t0:.0f}s)")
missing = [pt['tag'] for pt in ALL_PTS if pt['tag'] not in COVS]
GATES['G18_cov_binding'] = True
GATES['G19_cov_sha'] = True
GATES['G11_cov_psd'] = True
GATES['G09_symmetry_projection'] = bool(
    all(COVS[t][2]['symmetry']['reality_raw'] <= t1.RAW_SYMMETRY_CEILING
        and COVS[t][2]['symmetry']['herm_raw'] <= t1.RAW_SYMMETRY_CEILING
        and COVS[t][2]['symmetry']['reality_post'] < 1e-12
        and COVS[t][2]['projection_impact_ESpm'] < t1.PROJECTION_IMPACT_CEILING
        for t in COVS))
assert GATES['G09_symmetry_projection']
print('対称性射影サマリ: raw最大 reality={:.2e} herm={:.2e} / E[S±]影響最大 {:.2e}'.format(
    max(COVS[t][2]['symmetry']['reality_raw'] for t in COVS),
    max(COVS[t][2]['symmetry']['herm_raw'] for t in COVS),
    max(COVS[t][2]['projection_impact_ESpm'] for t in COVS)))
assert not missing, missing
print(f'covariance完了検査: {len(COVS)}/{len(ALL_PTS)} OK')

In [ ]:
# ---- 主解析：raw (S+,S-,A,rho)・演算子分離・argmin S+同一軸走査 ----
NREAL = 100 if T1_MODE == 'smoke' else 400
SCAN_NSIDE = 4 if T1_MODE == 'smoke' else 8
Vdirs, Pes = t1.scan_projectors(SCAN_NSIDE, 'refl')
print(f'scan directions: {len(Vdirs)} (nside={SCAN_NSIDE})')
rows = []; NPZ = {}; EIG = {}
for pi, pt in enumerate(POINTS):
    Mx, Cr, meta = COVS[pt['tag']]
    x, einfo = t1.sample_real(Cr, NREAL, seed=1000 * pi, return_info=True)
    EIG[pt['tag']] = einfo
    row = dict(tag=pt['tag'], topology=pt['topology'], nreal=NREAL,
               cov_array_sha=meta['cov_array_sha256'][:16])
    for (op, nm), nvec in AXES[pt['topology']].items():
        key = nm.replace('M_', '').replace('@', '') + ('_R' if op == 'refl' else '_H')
        Ep, Em = t1.exp_S(Cr, nvec, op)
        Sp, Sm, A, rho = t1.stats_at_axis(x, nvec, op)
        row[f'ESp_{key}'] = Ep; row[f'ESm_{key}'] = Em
        row[f'Sp_{key}'] = float(Sp.mean()); row[f'Sp_{key}_se'] = float(Sp.std() / np.sqrt(NREAL))
        row[f'Sm_{key}'] = float(Sm.mean()); row[f'Sm_{key}_se'] = float(Sm.std() / np.sqrt(NREAL))
        NPZ[f"{pt['tag']}|{key}"] = np.stack([Sp, Sm, A, rho])
        assert (abs(Sp.mean() - Ep) < 4 * Sp.std() / np.sqrt(NREAL) + 1e-9 * abs(Ep)
                and abs(Sm.mean() - Em) < 4 * Sm.std() / np.sqrt(NREAL) + 1e-9 * abs(Em)), \
            (pt['tag'], key, 'MC vs 解析期待値の不一致')
    sc = t1.scan_argmin_Splus(x, Vdirs, Pes)
    # 注（レビュー§12）：S++S-固定ゆえn*=argmin S+ではS-はscan選択で増大する。観測比較には
    # 観測側にも同一scan ruleを適用したjoint分布（Step 1）が必要で，この生値では比較しない。
    row['Splus_at_argmin'] = float(np.median(sc['Splus_at']))
    row['Sminus_at_argmin'] = float(np.median(sc['Sminus_at']))
    row['rho_at_argmin'] = float(np.median(sc['rho_at']))
    row['min_Sminus_diag'] = float(np.median(sc['min_Sminus']))
    NPZ[f"{pt['tag']}|scan"] = np.stack([sc['Splus_at'], sc['Sminus_at'], sc['A_at'], sc['rho_at']])
    rows.append(row)
    print(f"  {pt['tag']:20s} done")
df = pd.DataFrame(rows)
CSV = os.path.join(OUT, 't1_audited_results.csv'); df.to_csv(CSV, index=False)
np.savez_compressed(os.path.join(OUT, 't1_audited_realizations.npz'), **NPZ)
print('saved:', CSV)

In [ ]:
# ---- 等方収束gate（sanity convergence gate）：最終絶対条件・形状＋絶対振幅 ----
lmf_all = [(l, m) for l in (2, 3, 4) for m in range(-l, l + 1)]
C_isoW_c = np.diag(np.array([CL_CMBTOPO_REF[l] for (l, m) in lmf_all]).astype(complex))
iso_rep = []
for pt in ISO_SERIES:
    Mx, Cr, meta = COVS[pt['tag']]
    diag = np.real(np.diag(Mx))
    per = {l: np.mean([diag[i] for i, (li, _) in enumerate(lmf_all) if li == l])
           for l in (2, 3, 4)}
    rel = {l: float((per[l] / CL_CMBTOPO_REF[l]) / (per[2] / CL_CMBTOPO_REF[2]))
           for l in per}
    off = float(np.sum(np.abs(Mx - np.diag(np.diag(Mx))) ** 2) / np.sum(np.abs(Mx) ** 2))
    alpha = float(np.real(np.vdot(C_isoW_c, Mx)) / np.real(np.vdot(C_isoW_c, C_isoW_c)))
    d_iso = float(np.linalg.norm(Mx - alpha * C_isoW_c) / np.linalg.norm(alpha * C_isoW_c))
    rng = np.random.default_rng(0); sp = []
    for _ in range(6):
        n = rng.standard_normal(3); n /= np.linalg.norm(n)
        sp.append(t1.exp_S(Cr, n, 'refl'))
    sp = np.array(sp)
    ori = float(np.ptp(sp, axis=0).max() / sp.mean())
    iso_rep.append(dict(tag=pt['tag'], rel=rel, offdiag_frac=off, d_iso=d_iso,
                        orientation_spread=ori, alpha=alpha))
    print(f"  {pt['tag']:16s} rel={ {l: round(r, 3) for l, r in rel.items()} } "
          f"offdiag={off:.4f} d_iso={d_iso:.4f} ori={ori:.4f}")
fin = iso_rep[-1]
GATES['G22_iso_distance'] = bool(fin['d_iso'] < 0.25 and fin['offdiag_frac'] < 0.05
                                 and all(abs(r - 1) < 0.15 for r in fin['rel'].values()))
GATES['G22b_iso_amplitude'] = bool(abs(fin['alpha'] - 1) < 0.15)
GATES['G23_iso_orientation'] = bool(fin['orientation_spread'] < 0.02)
print(f"絶対振幅 alpha(最終L)={fin['alpha']:.4f}（G22b: |alpha-1|<0.15）")
trend_note = dict(offdiag=[r['offdiag_frac'] for r in iso_rep],
                  d_iso=[r['d_iso'] for r in iso_rep],
                  orientation=[r['orientation_spread'] for r in iso_rep])
print('収束系列（記録・単調性はhard条件にしない）:', trend_note)
assert (GATES['G22_iso_distance'] and GATES['G22b_iso_amplitude']
        and GATES['G23_iso_orientation']), \
    ('等方gate（sanity convergence gate）FAIL——事前ルール：thresholdを緩めず，より大きい'
     'L（5）を追加。全Lで安定した定数alpha!=1なら単位規約の相違＝official前に文書化・'
     '全体適用して再監査（rules v1.4 §4c）', fin)
ISO_GATE = dict(series=iso_rep, trend=trend_note)
print('=== ISO CONVERGENCE GATE PASS ===')

In [ ]:
# ---- 旧v0.3差分表（regression・独立validationではない）＋完全性gate＋維持/修正/撤回 ----
REF_V03 = {'E7_L1y1.0_g0': (1.013, 0.671), 'E7_L1y1.0_gq': (0.993, 0.693),
           'E7_L1y1.0_gh': (1.019, 0.691), 'E7_L1y0.85_g0': (1.073, 0.640),
           'E7_L1y0.85_gq': (1.077, 0.633), 'E7_L1y0.85_gh': (1.057, 0.654),
           'E7_L1y0.7_g0': (1.043, 0.672), 'E7_L1y0.7_gq': (1.069, 0.661),
           'E7_L1y0.7_gh': (1.090, 0.630), 'E7_L1y0.6_g0': (1.138, 0.568),
           'E7_L1y0.6_gq': (1.117, 0.600), 'E7_L1y0.6_gh': (1.115, 0.587),
           'E7_LAx0.7': (1.023, 0.714), 'E7_LAx1.3': (1.012, 0.676), 'E7_tilt': (0.996, 0.708)}
have = list(df['tag'])
assert len(have) == len(set(have)), 'df内の重複tag'
if T1_MODE == 'official':
    missing_reg = set(REF_V03) - set(have)
    GATES['G21_regression_complete'] = (len(missing_reg) == 0)
    assert GATES['G21_regression_complete'], f'regression対象の欠測: {missing_reg}'
else:
    GATES['G21_regression_complete'] = None
    print('（smoke：regression完全性gateはofficialで適用）')
print('※旧値は自己正規化比・新値はraw解析期待値——定義が異なる対照表（同一誤り共有の可能性が'
      'あるためregressionは独立validationとは呼ばない）')
print('tag                旧(Sp,Sm)      新E[Sp]/E[Sm]@A_R    符号主張(E[Sp]<E[Sm])  判定')
for _, r in df.iterrows():
    if r['tag'] not in REF_V03 or 'ESp_A_R' not in r or pd.isna(r.get('ESp_A_R')):
        continue
    sp, sm = REF_V03[r['tag']]
    Ep, Em = r['ESp_A_R'], r['ESm_A_R']
    v = '維持' if (Ep < Em) == (sp < sm) else '要検討'
    print(f"{r['tag']:18s} ({sp:.3f},{sm:.3f})  ({Ep:.1f},{Em:.1f})        "
          f"{str(Ep < Em):5s}                {v}")
SUMMARY = dict(
    maintained='符号比較の向き（E7系refl_A軸でE[S+]とE[S-]の大小）が新旧で一致するかを上表で判定',
    corrected='旧v0.3のMC絶対値（自己正規化比）は撤回し，本監査版のraw値（解析E[S±]＋MC分布）に置換。'
              'half-turn軸のmirror統計値は全て撤回（別演算子_Hで再定義）',
    retracted='sign map（S+<S-）を主判定とする研究設計を撤回。主役はjoint p(S+,S-|M)（Step 1で観測と'
              '同一統計・同一scan ruleの下で比較）')
print(json.dumps(SUMMARY, ensure_ascii=False, indent=1))

In [ ]:
# ---- 完了検査・provenance・OFFICIAL（G01–G25） ----
import scipy, camb as _cb, datetime
expected = {pt['tag'] for pt in POINTS}
GATES['G20_points_complete'] = (expected == set(df['tag']))
assert GATES['G20_points_complete'], (expected ^ set(df['tag']))
# G25：適用可能列のみのfinite検査（topology毎に演算子列が異なるための構造的NaNは許容）
BASE_COLS = ['Splus_at_argmin', 'Sminus_at_argmin', 'rho_at_argmin', 'min_Sminus_diag']
def applicable_cols(topology):
    cols = list(BASE_COLS)
    for (op, nm) in AXES[topology]:
        key = nm.replace('M_', '').replace('@', '') + ('_R' if op == 'refl' else '_H')
        cols += [f'ESp_{key}', f'ESm_{key}', f'Sp_{key}', f'Sp_{key}_se',
                 f'Sm_{key}', f'Sm_{key}_se']
    return cols
bad = []
for _, r in df.iterrows():
    for c in applicable_cols(r['topology']):
        v = r.get(c)
        if v is None or pd.isna(v) or not np.isfinite(v):
            bad.append((r['tag'], c))
GATES['G25_applicable_finite'] = (len(bad) == 0)
assert GATES['G25_applicable_finite'], bad[:10]
GATES['G25b_npz_finite'] = bool(all(np.isfinite(a).all() for a in NPZ.values()))
assert GATES['G25b_npz_finite']
core_gates = {k: v for k, v in GATES.items() if v is not None}
OFFICIAL = bool(T1_MODE == 'official' and all(core_gates.values()))
prov = dict(notebook='T1 signmap v1.5_audited', date=str(datetime.date.today()), mode=T1_MODE,
            status='OFFICIAL' if OFFICIAL else 'NON-OFFICIAL (smoke/pre-commit)',
            role=('retroactively audited low-l full-sky theoretical predictive-engine '
                  'validation; NOT a direct observational model comparison (Step 1)'),
            distribution_status='PILOT_DISTRIBUTION_ONLY / NOT_FOR_TAIL_PVALUES',
            supersedes='T1_signmap_v0.3 (EXPLORATORY/SUPERSEDED)',
            gates=GATES, repo_gate=REPO_GATE,
            engine_path_absolute=str(ENGINE_PATH),
            engine_file_sha256=hashlib.sha256(open(ENGINE_PATH, 'rb').read()).hexdigest(),
            t2b2_bridge_sha256=hashlib.sha256(
                open(os.path.join(MT_REPO, 't2b2_bridge.py'), 'rb').read()).hexdigest(),
            t2b2_run_sha256=hashlib.sha256(
                open(os.path.join(MT_REPO, 't2b2_run.py'), 'rb').read()).hexdigest(),
            cmbtopology=dict(commit=EXPECTED_CMBTOPO_COMMIT, origin=CT_ORIGIN,
                             camb=CT_CAMB, path=CT_DIR, ct_repair=CT_REPAIR),
            cl_reference=dict(audit=CL_AUDIT, cmbtopo_ref=CL_CMBTOPO_REF),
            validation=VALID_T1, iso_gate=ISO_GATE, eig_info=EIG,
            covariances={pt['tag']: dict(file=COVS[pt['tag']][2]['cov_file_sha256'],
                                         array=COVS[pt['tag']][2]['cov_array_sha256'],
                                         symmetry=COVS[pt['tag']][2]['symmetry'],
                                         projection_impact=COVS[pt['tag']][2]
                                         ['projection_impact_ESpm'],
                                         eig=COVS[pt['tag']][2]['eig'])
                         for pt in ALL_PTS},
            symmetry_ceilings=dict(raw=t1.RAW_SYMMETRY_CEILING,
                                   impact=t1.PROJECTION_IMPACT_CEILING),
            rng='numpy default_rng', seeds='per-point seed=1000*index; single generator',
            nreal=NREAL, scan_nside=SCAN_NSIDE,
            operator='numerical harmonic rotation v1.5 (direct-geometry validated)',
            versions=dict(python=sys.version.split()[0], numpy=np.__version__,
                          scipy=scipy.__version__, healpy=hp.__version__, camb=_cb.__version__),
            outputs=dict(
                csv_sha256=hashlib.sha256(open(CSV, 'rb').read()).hexdigest(),
                npz_sha256=hashlib.sha256(
                    open(os.path.join(OUT, 't1_audited_realizations.npz'), 'rb').read()
                ).hexdigest(),
                scan_dirs_sha256=hashlib.sha256(np.ascontiguousarray(Vdirs).tobytes()).hexdigest(),
                axes_manifest_sha256=hashlib.sha256(json.dumps(
                    {t: {f'{op}:{nm}': np.round(v, 12).tolist()
                         for (op, nm), v in a.items()}
                     for t, a in AXES.items()}, sort_keys=True).encode()).hexdigest()))
json.dump(prov, open(os.path.join(OUT, 't1_audited_provenance.json'), 'w'),
          indent=1, ensure_ascii=False)
GATES['G26_output_hashes'] = True
core_gates = {k: v for k, v in GATES.items() if v is not None}
OFFICIAL = bool(T1_MODE == 'official' and all(core_gates.values()))
prov['gates'] = GATES; prov['status'] = 'OFFICIAL' if OFFICIAL else prov['status']
json.dump(prov, open(os.path.join(OUT, 't1_audited_provenance.json'), 'w'),
          indent=1, ensure_ascii=False)
print('OFFICIAL =', OFFICIAL)
print('gates:', json.dumps({k: v for k, v in GATES.items()}, ensure_ascii=False))

## 実行手順（遡及監査プロトコル・レビュー§18準拠）
1. **3点をcommit＆push**：本ノートブック（v1.5）・`t1_engine.py`（v1.5）・
   `docs/T1_audit_rules_v1.4.md`。
2. Colabスクラッチコピーで`T1_MODE='smoke'`全実行→成果物返送→独立確認。
3. Colabスクラッチコピー（冒頭に`T1_MODE='smoke'`セルを追加）で全実行→smoke成果物返送→
   独立確認。
4. クローン内の純正ノートブックをRuntime restart→Run all（official）。
5. CSV・NPZ・provenance・全セル出力を返送→ChatGPT独立検算→freeze→次の旧プログラムへ。